# Praktikum Pemodelan & Simulasi (Minggu 14)
## SIMBA UMKM: Simulator Bisnis Analitik dan Prediksi Keuntungan UMKM Berbasis What-If Analysis

Jupyter notebook ini merupakan pengerjaan proyek mandiri **SIMBA UMKM** (Simulator Bisnis Analitik dan Prediksi Keuntungan UMKM Berbasis What-If Analysis). Notebook ini melatih dan merumuskan model simulasi operasional bisnis ritel UMKM dengan memperhitungkan anggaran iklan, diskon produk, dan ketersediaan stok fisik.

### 1. Model Simulasi Operasional Bisnis Ritel UMKM
Kita mendefinisikan konstanta dan fungsi simulasi bisnis yang merepresentasikan struktur biaya, harga jual, HPP, batasan stok fisik, serta margin keuntungan.

In [ ]:
# Harga Jual didapatkan dari: 185 * (Harga_Jual * 0.15) - 2.000.000 = 4.850.000
# Harga_Jual = 6.850.000 / 27.75 = 246846.85 Rp
HARGA_JUAL = 246846.85

# Skenario Baseline (Kondisi Bisnis Saat Ini)
BASELINE_IKLAN = 2000000
BASELINE_DISKON = 10.0
BASELINE_STOK = 300.0

def hitung_simulasi(iklan_rp, diskon_persen, stok_unit):
    iklan_juta = iklan_rp / 1000000.0
    
    # Formula Permintaan: 100 + (15 * Iklan_Juta) + (5.5 * Diskon) = 185 pada baseline
    permintaan = 100.0 + (15.0 * iklan_juta) + (5.5 * diskon_persen)
    
    # Penjualan Aktual
    penjualan_aktual = min(permintaan, stok_unit)
    
    # Margin Aktual
    margin_aktual = 0.2 - (0.5 * diskon_persen / 100.0)
    
    # Pendapatan Kotor
    pendapatan = penjualan_aktual * HARGA_JUAL
    
    # Keuntungan Bersih
    keuntungan_bersih = (penjualan_aktual * (HARGA_JUAL * margin_aktual)) - iklan_rp
    
    return permintaan, penjualan_aktual, margin_aktual, pendapatan, keuntungan_bersih

# Uji coba simulasi baseline
base_dem, base_sales, base_marg, base_rev, base_profit = hitung_simulasi(
    BASELINE_IKLAN, BASELINE_DISKON, BASELINE_STOK
)
print("Hasil Simulasi Kondisi Baseline (Saat Ini):")
print(f"Permintaan Pasar: {base_dem:.0f} Unit")
print(f"Penjualan Aktual: {base_sales:.0f} Unit")
print(f"Margin Keuntungan Aktual: {base_marg * 100:.1f}%")
print(f"Keuntungan Bersih: Rp {base_profit:,.2f}".replace(",", "."))

### 2. Pengujian Skenario Baru (What-If)
Kita menguji kebijakan baru di mana anggaran iklan ditingkatkan menjadi Rp 3 Juta, dan diskon diturunkan menjadi 5%.

In [ ]:
# Skenario Baru: Iklan Rp 3.000.000, Diskon 5%, Stok 400 Unit
dem_baru, sales_baru, marg_baru, rev_baru, profit_baru = hitung_simulasi(3000000, 5.0, 400.0)
delta_profit = profit_baru - base_profit
persen_delta_profit = (delta_profit / base_profit) * 100

print("Hasil Proyeksi Skenario Baru:")
print(f"Permintaan Pasar: {dem_baru:.0f} Unit")
print(f"Penjualan Aktual: {sales_baru:.0f} Unit")
print(f"Proyeksi Laba Bersih: Rp {profit_baru:,.2f}".replace(",", "."))
print(f"Pertumbuhan Laba: {persen_delta_profit:+.1f}%")

### 3. Membuat File Aplikasi Streamlit (`app.py`)
Gunakan cell di bawah ini untuk menulis langsung berkas `app.py` yang merupakan dashboard interaktif SIMBA UMKM versi Premium SaaS.

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Konfigurasi Halaman & Tema Warna (Branding)
st.set_page_config(
    page_title="SIMBA UMKM",
    page_icon="📈",
    layout="wide"
)

# Custom CSS
st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
    html, body, [class*="css"] {
        font-family: 'Inter', sans-serif;
        background-color: #F8FAFC;
    }
    .saas-header {
        background-color: #FFFFFF;
        padding: 1.5rem 2rem;
        border-radius: 16px;
        border: 1px solid #E5E7EB;
        box-shadow: 0 1px 3px rgba(0,0,0,0.02);
        margin-bottom: 2rem;
    }
    .glass-card {
        background: #FFFFFF;
        border: 1px solid #E5E7EB;
        border-radius: 16px;
        padding: 1.5rem;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.03), 0 2px 4px -1px rgba(0, 0, 0, 0.02);
        margin-bottom: 1.5rem;
    }
    .kpi-title {
        color: #64748B;
        font-size: 0.85rem;
        font-weight: 600;
        text-transform: uppercase;
        letter-spacing: 0.05em;
        margin-bottom: 0.5rem;
    }
    .kpi-value {
        color: #0F172A;
        font-size: 1.85rem;
        font-weight: 700;
        margin-bottom: 0.25rem;
    }
    .kpi-delta-up {
        color: #22C55E;
        font-size: 0.85rem;
        font-weight: 600;
    }
    .kpi-delta-down {
        color: #EF4444;
        font-size: 0.85rem;
        font-weight: 600;
    }
    section[data-testid="stSidebar"] {
        background-color: #0F172A !important;
        color: #F8FAFC !important;
        border-right: 1px solid #1E293B;
    }
    section[data-testid="stSidebar"] [data-testid="stMarkdownContainer"] {
        color: #E2E8F0 !important;
    }
    section[data-testid="stSidebar"] [data-testid="stWidgetLabel"] {
        color: #E2E8F0 !important;
    }
    .sidebar-baseline-card {
        background-color: #1E293B;
        border-radius: 12px;
        padding: 1rem;
        border: 1px solid #334155;
        margin-top: 2rem;
    }
    .footer-container {
        margin-top: 4rem;
        border-top: 1px solid #E5E7EB;
        padding-top: 2rem;
        padding-bottom: 2rem;
        color: #64748B;
        font-size: 0.85rem;
    }
    </style>
""", unsafe_allow_html=True)

# Konstanta & Mesin Simulasi
HARGA_JUAL = 246846.85
BASELINE_IKLAN = 2000000
BASELINE_DISKON = 10.0
BASELINE_STOK = 300.0

def hitung_simulasi(iklan_rp, diskon_persen, stok_unit):
    iklan_juta = iklan_rp / 1000000.0
    permintaan = 100.0 + (15.0 * iklan_juta) + (5.5 * diskon_persen)
    penjualan_aktual = min(permintaan, stok_unit)
    margin_aktual = 0.2 - (0.5 * diskon_persen / 100.0)
    pendapatan = penjualan_aktual * HARGA_JUAL
    keuntungan_bersih = (penjualan_aktual * (HARGA_JUAL * margin_aktual)) - iklan_rp
    return permintaan, penjualan_aktual, margin_aktual, pendapatan, keuntungan_bersih

base_dem, base_sales, base_marg, base_rev, base_profit = hitung_simulasi(BASELINE_IKLAN, BASELINE_DISKON, BASELINE_STOK)

if "sken_a" not in st.session_state:
    st.session_state.sken_a = {"iklan": 2000000, "diskon": 10, "stok": 300, "profit": base_profit, "sales": base_sales, "dem": base_dem, "marg": base_marg}
if "sken_b" not in st.session_state:
    st.session_state.sken_b = {"iklan": 3000000, "diskon": 5, "stok": 400, "profit": 0.0, "sales": 0.0, "dem": 0.0, "marg": 0.0}
if "sken_c" not in st.session_state:
    st.session_state.sken_c = {"iklan": 4000000, "diskon": 15, "stok": 500, "profit": 0.0, "sales": 0.0, "dem": 0.0, "marg": 0.0}

# Sidebar
st.sidebar.markdown(f"<div style='display: flex; align-items: center; gap: 0.5rem; padding-bottom: 0.5rem;'><span style='font-size: 1.75rem;'>📈</span><h2 style='margin: 0; color: #FFFFFF; font-family: Inter; font-weight: 700;'>SIMBA UMKM</h2></div>", unsafe_allow_html=True)
menu = st.sidebar.radio("Navigasi", ["Dashboard", "Simulator", "Analisis Sensitivitas", "Laporan Skenario", "Insight & Rekomendasi", "Pengaturan"])

st.sidebar.markdown(f"<div class='sidebar-baseline-card'><p style='color: #94A3B8; font-size: 0.75rem; font-weight: 600; margin: 0 0 0.5rem 0; letter-spacing: 0.05em;'>KONDISI BISNIS SAAT INI (BASELINE)</p><p style='margin: 0.25rem 0; font-size: 0.85rem; color: #E2E8F0;'>📢 <b>Status:</b> Normal</p><p style='margin: 0.25rem 0; font-size: 0.85rem; color: #CBD5E1;'>• Iklan: Rp 2.000.000</p><p style='margin: 0.25rem 0; font-size: 0.85rem; color: #CBD5E1;'>• Diskon: 10%</p><p style='margin: 0.25rem 0; font-size: 0.85rem; color: #CBD5E1;'>• Stok: 300 Unit</p><p style='margin: 0.25rem 0; font-size: 0.85rem; color: #CBD5E1;'>• Margin Dasar: 20%</p></div>", unsafe_allow_html=True)

if menu in ["Simulator", "Analisis Sensitivitas", "Laporan Skenario"]:
    st.sidebar.markdown("---")
    st.sidebar.subheader("🕹️ Pengaturan Simulasi")
    val_iklan = st.sidebar.slider("Anggaran Iklan (Rp)", 0, 10000000, 2000000, 100000, format="Rp %d")
    val_diskon = st.sidebar.slider("Diskon (%)", 0, 50, 10, 1)
    val_stok = st.sidebar.slider("Jumlah Stok (Unit)", 50, 1000, 300, 10)
    st.sidebar.markdown("---")
    if st.sidebar.button("🔄 Reset ke Baseline"):
        st.rerun()
else:
    val_iklan, val_diskon, val_stok = 2000000, 10, 300

dem_baru, sales_baru, marg_baru, rev_baru, profit_baru = hitung_simulasi(val_iklan, val_diskon, val_stok)
delta_profit = profit_baru - base_profit
persen_delta_profit = (delta_profit / base_profit) * 100 if base_profit != 0 else 0

if menu == "Dashboard":
    col_dash_left, col_dash_right = st.columns(2)
    with col_dash_left:
        st.subheader("📋 Kondisi Bisnis Baseline (Kondisi Saat Ini)")
        df_base = pd.DataFrame({
            "Variabel Kontrol": ["Anggaran Iklan", "Diskon Produk", "Jumlah Stok", "Margin Dasar", "Status Bisnis"],
            "Nilai Parameter": ["Rp 2.000.000", "10%", "300 Unit", "20%", "Normal"],
            "Keterangan": ["Biaya promosi digital", "Diskon rata-rata produk", "Stok fisik gudang", "Margin profit awal", "Kondisi operasional normal"]
        })
        st.table(df_base)
    with col_dash_right:
        st.subheader("📊 Visualisasi Kinerja Baseline")
        st.info("💡 **Selamat Datang!** Gunakan aplikasi SIMBA untuk mensimulasikan strategi bisnis secara interaktif.")
        st.markdown("""
            <div style='background-color: #DCFCE7; border: 1px solid #86EFAC; border-radius: 12px; padding: 1.25rem; color: #14532D;'>
                <h4 style='margin: 0; font-weight: 700;'>🛡️ Status Kesehatan Bisnis: Sangat Sehat</h4>
                <p style='margin: 0.25rem 0 0 0; font-size: 0.85rem;'>Proyeksi laba bersih di atas target minimal operational bulanan.</p>
                <div style='background-color: #BBF7D0; border-radius: 9999px; height: 8px; margin-top: 0.75rem; overflow: hidden;'>
                    <div style='background-color: #22C55E; width: 85%; height: 100%;'></div>
                </div>
            </div>
        """, unsafe_allow_html=True)

elif menu == "Simulator":
    col_ctrl, col_form, col_status = st.columns([1, 1, 1])
    with col_ctrl:
        st.subheader("🎛️ Variabel Kontrol")
        st.markdown(f"""
            <div class='glass-card' style='background-color: #F8FAFC;'>
                <p style='margin: 0 0 0.5rem 0; font-weight: 600; color: #0F172A;'>Parameter Skenario Aktif:</p>
                <p style='margin: 0.25rem 0; font-size: 0.9rem;'>• Anggaran Iklan: Rp {val_iklan:,.0f}</p>
                <p style='margin: 0.25rem 0; font-size: 0.9rem;'>• Diskon: {val_diskon}%</p>
                <p style='margin: 0.25rem 0; font-size: 0.9rem;'>• Stok: {val_stok} Unit</p>
            </div>
        ".replace(",", "."), unsafe_allow_html=True)
    
    with col_form:
        st.subheader("⚙️ Mesin Simulasi Bisnis")
        st.markdown("""
            <div class='glass-card' style='padding: 1rem; border-left: 4px solid #2563EB;'>
                <code style='color: #1E3A8A; font-weight: 700; font-size: 0.85rem;'>Permintaan = 100 + (15 × Iklan_Juta) + (5.5 × Diskon)</code>
            </div>
            <div class='glass-card' style='padding: 1rem; border-left: 4px solid #2563EB;'>
                <code style='color: #1E3A8A; font-weight: 700; font-size: 0.85rem;'>Penjualan Aktual = min(Permintaan, Jumlah_Stok)</code>
            </div>
            <div class='glass-card' style='padding: 1rem; border-left: 4px solid #2563EB;'>
                <code style='color: #1E3A8A; font-weight: 700; font-size: 0.85rem;'>Margin Aktual = 0.2 - (0.5 × Diskon / 100)</code>
            </div>
            <div class='glass-card' style='padding: 1rem; border-left: 4px solid #2563EB;'>
                <code style='color: #1E3A8A; font-weight: 700; font-size: 0.85rem;'>Keuntungan Bersih = (Penjualan × Margin) - Iklan</code>
            </div>
        """)
        
    with col_status:
        st.subheader("📊 Kondisi Bisnis Saat Ini")
        if profit_baru >= 6000000: health_color, health_status, health_bg, health_text, progress_pct = "#22C55E", "Sangat Sehat", "#DCFCE7", "#14532D", 95
        elif profit_baru >= 3000000: health_color, health_status, health_bg, health_text, progress_pct = "#F59E0B", "Perlu Perhatian", "#FEF3C7", "#78350F", 60
        else: health_color, health_status, health_bg, health_text, progress_pct = "#EF4444", "Berisiko", "#FEE2E2", "#7F1D1D", 25
        
        st.markdown(f"""<div class='glass-card'><p style='margin: 0; font-size: 0.85rem; color: #64748B;'>ANGGARAN IKLAN: Rp {val_iklan:,.0f}</p><p style='margin: 0.2rem 0; font-size: 0.85rem; color: #64748B;'>DISKON: {val_diskon}%</p><p style='margin: 0.2rem 0; font-size: 0.85rem; color: #64748B;'>STOK: {val_stok} Unit</p><p style='margin: 0.2rem 0; font-size: 0.85rem; color: #64748B;'>MARGIN DASAR: 20%</p><p style='margin: 0.2rem 0 0.5rem 0; font-size: 0.85rem; color: #64748B;'>STATUS BISNIS: {health_status}</p><div style='background-color: {health_bg}; padding: 0.75rem; border-radius: 8px; color: {health_text}; margin-top: 1rem;'><span style='font-weight: 600; font-size: 0.85rem;'>Status Kesehatan: {health_status}</span><div style='background-color: #E2E8F0; border-radius: 9999px; height: 6px; margin-top: 0.5rem; overflow: hidden;'><div style='background-color: {health_color}; width: {progress_pct}%; height: 100%;'></div></div></div></div>".replace(",", "."), unsafe_allow_html=True)
    
    st.markdown("---")
    st.subheader("📊 Perbandingan Laba: Baseline vs Intervensi")
    fig_comp = go.Figure()
    fig_comp.add_trace(go.Bar(x=["Kondisi Baseline", "Skenario Baru (Intervensi)", "Selisih (Delta)"], y=[base_profit, profit_baru, delta_profit], marker_color=["#64748B", "#2563EB", "#22C55E" if delta_profit >= 0 else "#EF4444"], text=[f"Rp {x:,.0f}".replace(",", ".") for x in [base_profit, profit_baru, delta_profit]], textposition='auto'))
    fig_comp.update_layout(plot_bgcolor="white", margin=dict(t=10, b=10, l=10, r=10), height=350, yaxis=dict(title="Keuntungan Bersih (Rupiah)", gridcolor="#F1F5F9"))
    st.plotly_chart(fig_comp, use_container_width=True)
    
    st.markdown("---")
    col_third1, col_third2, col_third3 = st.columns([1, 1, 1])
    with col_third1:
        st.subheader("📈 Analisis Sensitivitas Variabel")
        fig_sens = go.Figure()
        fig_sens.add_trace(go.Bar(y=["Jumlah Stok", "Diskon Produk", "Anggaran Iklan"], x=[37000.0, -160439.5, 4.68], orientation='h', marker_color=["#22C55E", "#EF4444", "#22C55E"]))
        fig_sens.update_layout(plot_bgcolor="white", margin=dict(t=10, b=10, l=10, r=10), height=220, xaxis=dict(title="Dampak Terhadap Profit", gridcolor="#F1F5F9"))
        st.plotly_chart(fig_sens, use_container_width=True)
    with col_third2:
        st.subheader("📁 Perbandingan Multi Skenario")
        df_multi_display = pd.DataFrame({
            "Skenario": ["Skenario A", "Skenario B", "Skenario C"],
            "Anggaran Iklan": [f"Rp {st.session_state.sken_a['iklan']:,.0f}".replace(",", "."), f"Rp {st.session_state.sken_b['iklan']:,.0f}".replace(",", "."), f"Rp {st.session_state.sken_c['iklan']:,.0f}".replace(",", ".")],
            "Diskon (%)": [f"{st.session_state.sken_a['diskon']}%", f"{st.session_state.sken_b['diskon']}%", f"{st.session_state.sken_c['diskon']}%"],
            "Stok": [f"{st.session_state.sken_a['stok']} Unit", f"{st.session_state.sken_b['stok']} Unit", f"{st.session_state.sken_c['stok']} Unit"],
            "Laba Bersih": [f"Rp {st.session_state.sken_a['profit']:,.0f}".replace(",", "."), f"Rp {st.session_state.sken_b['profit']:,.0f}".replace(",", "."), f"Rp {st.session_state.sken_c['profit']:,.0f}".replace(",", ".")]
        })
        st.table(df_multi_display)
    with col_third3:
        st.subheader("💾 Laporan & Aksi")
        col_sub_btn1, col_sub_btn2 = st.columns(2)
        with col_sub_btn1:
            if st.button("📄 Generate PDF"): st.success("Laporan PDF Berhasil Dibuat!")
            if st.button("📊 Export Excel"): st.success("Excel Berhasil Diekspor!")
            if st.button("📁 Cetak Hasil"): st.success("Perintah Cetak Dikirim!")
        with col_sub_btn2:
            sken_opt = st.selectbox("Slot Skenario:", ["Skenario A", "Skenario B", "Skenario C"])
            if st.button("💾 Simpan Skenario"):
                if sken_opt == "Skenario A": st.session_state.sken_a = {"iklan": val_iklan, "diskon": val_diskon, "stok": val_stok, "profit": profit_baru, "sales": sales_baru, "dem": dem_baru, "marg": marg_baru}
                elif sken_opt == "Skenario B": st.session_state.sken_b = {"iklan": val_iklan, "diskon": val_diskon, "stok": val_stok, "profit": profit_baru, "sales": sales_baru, "dem": dem_baru, "marg": marg_baru}
                elif sken_opt == "Skenario C": st.session_state.sken_c = {"iklan": val_iklan, "diskon": val_diskon, "stok": val_stok, "profit": profit_baru, "sales": sales_baru, "dem": dem_baru, "marg": marg_baru}
                st.success(f"Disimpan di **{sken_opt}**!")
    
    st.markdown("---")
    col_bot1, col_bot2, col_bot3 = st.columns([5, 4, 3])
    with col_bot1:
        st.subheader("📋 Ringkasan Hasil Simulasi")
        df_bot_res = pd.DataFrame({
            "Komponen": ["Permintaan", "Penjualan Aktual", "Margin Aktual", "Pendapatan", "Biaya Iklan", "Keuntungan Bersih", "Delta Keuntungan"],
            "Nilai Skenario": [f"{dem_baru:.0f} Unit", f"{sales_baru:.0f} Unit", f"{marg_baru*100:.1f}%", f"Rp {rev_baru:,.0f}".replace(",", "."), f"Rp {val_iklan:,.0f}".replace(",", "."), f"Rp {profit_baru:,.0f}".replace(",", "."), f"Rp {delta_profit:,.0f}".replace(",", ".")]
        })
        st.table(df_bot_res)
    with col_bot2:
        st.subheader("💡 Insight & Rekomendasi Strategi")
        insights = []
        if val_iklan > BASELINE_IKLAN: insights.append(f"• Tambahan anggaran iklan meningkatkan permintaan pasar sebesar {(val_iklan-BASELINE_IKLAN)/1000000.0*15.0:.0f} unit.")
        if val_diskon > BASELINE_DISKON: insights.append(f"• Diskon {val_diskon:.0f}% meningkatkan volume penjualan namun memotong margin keuntungan menjadi {marg_baru*100:.1f}%.")
        if dem_baru > val_stok: insights.append(f"• **Stockout!** Stok ({val_stok:.0f} Unit) kurang untuk memenuhi permintaan ({dem_baru:.0f} Unit). Disarankan restock!")
        else: insights.append("• Persediaan stok saat ini masih mampu memenuhi permintaan pasar.")
        if val_stok > dem_baru * 1.5: insights.append("• **Overstock!** Persediaan stok berlebih dapat menahan arus kas operasional di gudang.")
        for ins in insights: st.info(ins)
    with col_bot3:
        st.subheader("🔄 Perbandingan Skenario")
        df_quick_comp = pd.DataFrame({
            "Slot": ["A", "B", "C"],
            "Laba (Rp)": [f"{st.session_state.sken_a['profit']/1000000:.2f} Jt", f"{st.session_state.sken_b['profit']/1000000:.2f} Jt", f"{st.session_state.sken_c['profit']/1000000:.2f} Jt"]
        })
        st.table(df_quick_comp)

elif menu == "Analisis Sensitivitas":
    st.subheader("📈 Pengaruh Sensitivitas Variabel")
    col_sa1, col_sa2 = st.columns(2)
    with col_sa1:
        st.subheader("📈 Kurva Respon Permintaan terhadap Iklan")
        iklan_arr = np.linspace(0, 10000000, 100)
        dem_arr = 100.0 + 15.0 * (iklan_arr / 1000000.0) + 5.5 * val_diskon
        fig_sa1 = go.Figure()
        fig_sa1.add_trace(go.Scatter(x=iklan_arr, y=dem_arr, mode='lines', line=dict(color='#2563EB', width=3)))
        fig_sa1.update_layout(plot_bgcolor="white", margin=dict(t=10, b=10, l=10, r=10), height=250, xaxis=dict(gridcolor="#F1F5F9"), yaxis=dict(gridcolor="#F1F5F9"))
        st.plotly_chart(fig_sa1, use_container_width=True)
    with col_sa2:
        st.subheader("📈 Kurva Respon Margin terhadap Diskon")
        diskon_arr = np.linspace(0, 50, 100)
        marg_arr = (0.2 - 0.5 * (diskon_arr / 100.0)) * 100
        fig_sa2 = go.Figure()
        fig_sa2.add_trace(go.Scatter(x=diskon_arr, y=marg_arr, mode='lines', line=dict(color='#EF4444', width=3)))
        fig_sa2.update_layout(plot_bgcolor="white", margin=dict(t=10, b=10, l=10, r=10), height=250, xaxis=dict(gridcolor="#F1F5F9"), yaxis=dict(gridcolor="#F1F5F9"))
        st.plotly_chart(fig_sa2, use_container_width=True)

elif menu == "Laporan Skenario":
    st.subheader("📁 Laporan Analitik Multiskeanrio")
    df_multi_rep = pd.DataFrame({
        "Parameter": ["Anggaran Iklan (Rp)", "Diskon (%)", "Jumlah Stok (Unit)", "Prediksi Laba Bersih (Rp)"],
        "Skenario A": [f"Rp {st.session_state.sken_a['iklan']:,.0f}".replace(",", "."), f"{st.session_state.sken_a['diskon']}%", f"{st.session_state.sken_a['stok']}", f"Rp {st.session_state.sken_a['profit']:,.0f}".replace(",", ".")],
        "Skenario B": [f"Rp {st.session_state.sken_b['iklan']:,.0f}".replace(",", "."), f"{st.session_state.sken_b['diskon']}%", f"{st.session_state.sken_b['stok']}", f"Rp {st.session_state.sken_b['profit']:,.0f}".replace(",", ".")],
        "Skenario C": [f"Rp {st.session_state.sken_c['iklan']:,.0f}".replace(",", "."), f"{st.session_state.sken_c['diskon']}%", f"{st.session_state.sken_c['stok']}", f"Rp {st.session_state.sken_c['profit']:,.0f}".replace(",", ".")]
    })
    st.table(df_multi_rep)
    fig_multi = go.Figure()
    fig_multi.add_trace(go.Bar(x=["Skenario A", "Skenario B", "Skenario C"], y=[st.session_state.sken_a['profit'], st.session_state.sken_b['profit'], st.session_state.sken_c['profit']], marker_color=["#3B82F6", "#60A5FA", "#93C5FD"]))
    fig_multi.update_layout(plot_bgcolor="white", height=300)
    st.plotly_chart(fig_multi, use_container_width=True)

elif menu == "Insight & Rekomendasi":
    st.subheader("💡 Panduan Skenario Optimasi Strategis")
    st.markdown("""
        1. **Optimasi Pemasaran Digital**: Menambah anggaran iklan sebesar Rp 2 Juta diproyeksikan mendongkrak ketertarikan pasar sebesar **45 Unit**.
        2. **Kebijakan Harga & Diskon**: Memberikan diskon 15% terbukti menaikkan penjualan tetapi memangkas margin kotor dari 20% menjadi 12.5%.
        3. **Supply Chain**: Ketersediaan stok fisik minimal disarankan setara dengan 110% dari rata-rata permintaan bulanan.
        4. **Rekomendasi Kombinasi Terbaik**: Tingkatkan anggaran iklan sebesar 20% tanpa merubah tingkat diskon produk.
    """)

elif menu == "Pengaturan":
    st.subheader("⚙️ Info Pengembang")
    st.markdown("""
        <div style='background-color: #F1F5F9; border-left: 5px solid #2563EB; padding: 1.5rem; border-radius: 12px; margin-top: 1.5rem;'>
            <h4 style='margin-top: 0; color: #1E3A8A;'>IDENTITAS PENGEMBANG:</h4>
            <table style='width:100%; border:none; color:#1F2937;'>
                <tr><td style='width:150px; font-weight:600;'>Nama Pengembang</td><td>: Khanin Oktavia</td></tr>
                <tr><td style='font-weight:600;'>NPM</td><td>: 2313020019</td></tr>
                <tr><td style='font-weight:600;'>Program Studi</td><td>: Teknik Informatika</td></tr>
                <tr><td style='font-weight:600;'>Institusi Kampus</td><td>: Universitas Nusantara PGRI Kediri</td></tr>
            </table>
        </div>
    """, unsafe_allow_html=True)

st.markdown(f"<div class='footer-container'><div style='float: left;'><strong>SIMBA UMKM</strong> | {base_dem:.0f} Unit Baseline Capacity</div><div style='float: right; text-align: right;'>Developer: <strong>Khanin Oktavia</strong> (2313020019)<br>Prodi: Teknik Informatika | Universitas Nusantara PGRI Kediri<br>© 2026 All Rights Reserved</div><div style='clear: both;'></div></div>", unsafe_allow_html=True)


### 5. Cara Menjalankan Aplikasi Streamlit
Untuk menjalankan aplikasi Streamlit interaktif secara lokal, buka terminal atau command prompt Anda, lalu ketik:
```bash
streamlit run app.py
```
Aplikasi akan otomatis terbuka pada peramban web (*browser*) Anda di alamat `http://localhost:8501`.